# Market Foundation Model — Google Colab (T4)
Notebook resmi untuk menjalankan pipeline dari data mentah sampai fine-tuning. Jalankan berurutan. Runtime GPU T4 direkomendasikan; data dan checkpoint disimpan di Google Drive agar tidak hilang saat runtime terputus.

**Protokol:** tokenizer/pretraining/fine-tuning 2015–2020, pemilihan checkpoint 2021, kalibrasi 2022, final test 2023. Jangan mengubah split setelah melihat hasil final test.

> Catatan: training tidak menjamin profit. Periksa kualitas data, missing years, biaya transaksi, dan hasil out-of-sample.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/market-foundation-model')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NodePro-gif/market-foundation-model.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
%cd /content/market-foundation-model
print('Repository ready:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 1. GPU dan dependensi
Pilih **Runtime → Change runtime type → T4 GPU** sebelum menjalankan sel ini.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak aktif. Pilih runtime T4 GPU lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
!pip -q install -r requirements.txt


## 2. Hubungkan Google Drive
Data mentah, tokenizer, checkpoint dan hasil fine-tuning akan disimpan di `MyDrive/market-foundation-model`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/market-foundation-model')
STORE.mkdir(parents=True, exist_ok=True)
for name in ['corpus', 'npz', 'mkt', 'ckpt', 'ftune']:
    link = Path('/content') / name
    target = STORE / name
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    os.symlink(target, link, target_is_directory=True)
print('Persistent storage:', STORE)


## 3. Unduh dan verifikasi data
Tahap ini mengunduh OHLCV M5 untuk seluruh aset dan M1 untuk XAUUSD/EURUSD dari penyedia data yang dikonfigurasi di `stage1_corpus.py`. File yang tidak tersedia akan dilaporkan; periksa manifest sebelum melanjutkan.

In [ ]:
%cd /content/market-foundation-model
!python -u stage1_corpus.py


## 4. Bangun tokenizer dan label
Tahap ini CPU/RAM-intensive dan dapat memerlukan waktu. Statistik kuantil, regime, dan BPE hanya dipelajari dari periode training 2015–2020 dan aset non-held-out.

In [ ]:
!python -u test_tokenizer.py
!python -u stage2_tokenizer.py
!ls -lh /content/mkt | head -30


## 5. Pretraining GPT
Default T4: micro-batch 32 × accumulation 2 (effective batch 64), FP16 AMP, 6.000 optimizer steps. `resume.pt` disimpan berkala. Jika runtime terputus, jalankan ulang sel ini dengan `PRETRAIN_RESUME=1`; resume melanjutkan bobot dan optimizer, tetapi DataLoader memulai ulang epoch berjalan sehingga beberapa batch dapat terulang.

In [ ]:
import os
os.environ['PRETRAIN_BATCH'] = '32'
os.environ['PRETRAIN_ACCUM'] = '2'
os.environ['PRETRAIN_STEPS'] = '6000'
os.environ['PRETRAIN_CKPT_EVERY'] = '250'
os.environ['PRETRAIN_EVAL_EVERY'] = '250'
os.environ['PRETRAIN_WORKERS'] = '2'
os.environ['PRETRAIN_RESUME'] = '1'
!python -u stage3_pretrain.py


## 6. Verifikasi checkpoint dan tokenizer
Pemeriksaan ini menggunakan **validasi 2021**, bukan final test 2023. Fine-tuning akan menolak checkpoint tanpa manifest tokenizer yang cocok.

In [ ]:
!python -u verify_ckpt.py 50


## 7. Fine-tuning multi-task
Default T4: micro-batch 64 × accumulation 4 (effective batch 256), AMP FP16. Checkpoint dipilih hanya menggunakan AUC validasi 2021. Dataset final test 2023 baru dibuat setelah checkpoint terkunci.

In [ ]:
import os
os.environ['ST4_EPOCHS'] = '3'
os.environ['ST4_BATCH'] = '64'
os.environ['ST4_ACCUM'] = '4'
os.environ['ST4_LR'] = '1e-4'
os.environ['ST4_STRIDE'] = '4'
os.environ['ST4_VAL_STRIDE'] = '4'
os.environ['ST4_TEST_STRIDE'] = '4'
os.environ['ST4_HELD_STRIDE'] = '8'
os.environ['ST4_AMP'] = '1'
!python -u stage4_finetune.py


## 8. Kalibrasi dan backtest
Jalankan kalibrasi hanya pada 2022, lalu backtest pada 2023 dengan biaya transaksi realistis. Pastikan skrip evaluasi menggunakan checkpoint baru dari sesi ini.

In [ ]:
!python -u validate_stage4.py
!python -u backtest_stage4.py


## Lokasi artefak
- Data mentah: `MyDrive/market-foundation-model/corpus`
- Data hasil parsing: `npz/`
- Tokenizer dan label: `mkt/`
- Checkpoint dan resume: `ckpt/`
- Hasil fine-tuning: `ftune/`

Simpan commit SHA, konfigurasi, versi GPU/CUDA, jumlah data, dan asumsi spread/komisi/slippage bersama setiap hasil eksperimen.